# 01 · EDA and Preprocessing — The Engagement Trap

**Dataset:** MovieLens *ml-latest-small* (GroupLens, University of Minnesota): about 100,000 ratings (0.5–5 stars in half-star steps) by 610 users on about 9,700 movies, collected 1996–2018. Files: `ratings.csv` (userId, movieId, rating, timestamp), `movies.csv` (movieId, title with year, pipe-separated genres), plus `tags.csv` and `links.csv` (not used).

This notebook:
1. looks at the **raw** data, especially the **long tail** of movie popularity,
2. explains the preprocessing (filtering rarely-rated movies, re-indexing ids, genre and year features),
3. explores the filtered data used by all models (ratings, genres, years, time),
4. builds the content features for the PCA model and checks the train / validation / test split,
5. exports `movies.json` for the demo site.

The long tail is the motivation for the whole project. If a few movies already get most of the attention, a recommender trained on that data will tend to show mostly those movies, and a click metric won't tell you that it is doing so.

In [ ]:
import os, sys, glob, json, time
# On Kaggle, etlib.py comes from an attached dataset; locally it sits next to this notebook.
for hit in glob.glob("/kaggle/input/**/etlib.py", recursive=True):
    sys.path.insert(0, os.path.dirname(hit))
    break
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import etlib as et

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
OUT = et.output_dir()
DATA_DIR = et.find_data_dir()
ratings, items, GENRES = et.load_movielens(DATA_DIR)  # movies with >= 5 ratings
G = et.genre_matrix(items, GENRES)
print("data:", DATA_DIR, "| outputs:", OUT)
print(len(ratings), "ratings |", ratings.user.nunique(), "users |", ratings.item.nunique(), "movies")


def save_fig(fig, name):
    fig.savefig(f"{OUT}/figures/{name}.png", dpi=150, bbox_inches="tight")

## 1. Raw data and the long tail

The **Gini coefficient** measures inequality: 0 means every movie has the same number of ratings, values near 1 mean almost all ratings go to a handful of movies. The **Lorenz curve** plots the share of all ratings received by the least-popular x% of movies.

In [ ]:
raw_ratings, raw_movies = et.load_raw(DATA_DIR)
raw_counts = raw_ratings.movieId.value_counts()
print("raw ratings :", len(raw_ratings))
print("raw users   :", raw_ratings.userId.nunique())
print("movies in movies.csv:", len(raw_movies), "| movies with >= 1 rating:", len(raw_counts))
for t in (1, 2, 5, 10, 50, 100):
    print(f"movies with >= {t:3d} ratings: {(raw_counts >= t).sum():5d}  "
          f"({(raw_counts >= t).mean():.1%} of rated movies)")

In [ ]:
counts_sorted = raw_counts.sort_values(ascending=False).to_numpy()
top10_share = counts_sorted[: len(counts_sorted) // 10].sum() / counts_sorted.sum()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(counts_sorted)
axes[0].set(yscale="log", xlabel="movie rank", ylabel="ratings (log)", title="Ratings per movie, sorted (raw)")
axes[1].hist(raw_ratings.groupby("userId").size(), bins=60, color="darkorange")
axes[1].set(xlabel="ratings per user", ylabel="users", yscale="log", title="Ratings per user (raw)")
x = np.sort(counts_sorted)
axes[2].plot(np.linspace(0, 1, len(x)), np.cumsum(x) / x.sum(), label=f"movies (Gini = {et.gini(x):.3f})")
axes[2].plot([0, 1], [0, 1], "k--", label="perfect equality")
axes[2].set(xlabel="share of movies (least popular first)", ylabel="share of ratings", title="Lorenz curve (raw)")
axes[2].legend()
save_fig(fig, "01_long_tail")
plt.show()
print(f"The top 10% of movies receive {top10_share:.1%} of all ratings.")
summary = {"raw_ratings": len(raw_ratings), "raw_movies_rated": len(raw_counts),
           "raw_movies_with_1_rating": int((raw_counts == 1).sum()),
           "raw_top10pct_share": top10_share, "raw_gini_item_popularity": et.gini(x)}

## 2. Preprocessing

Done in `etlib.load_movielens` (read it alongside this notebook):
- **Filter rarely-rated movies.** Movies with fewer than **5** ratings are dropped. A movie rated once or twice has no reliable similarity (KNN) or embedding (SVD), and cannot be evaluated meaningfully. This keeps about 90% of the ratings, and the long tail is still very steep among the movies that remain.
- **Re-index ids.** userId / movieId are mapped to 0..n−1 so they can index matrix rows and columns directly. The original `movieId` is kept in `items`.
- **Genres.** The pipe-separated genre string becomes one 0/1 column per genre (including "(no genres listed)").
- **Year.** Parsed from the "(yyyy)" at the end of the title; a few titles have none.

Checks on the filtered data:

In [ ]:
per_user = ratings.groupby("user").size()
per_item = ratings.groupby("item").size()
summary.update({
    "n_ratings": len(ratings),
    "n_users": et.N_USERS,
    "n_items": et.N_ITEMS,
    "sparsity": 1 - len(ratings) / (et.N_USERS * et.N_ITEMS),
    "mean_rating": ratings.rating.mean(),
    "share_rated_4_or_more": (ratings.rating >= 4).mean(),
    "ratings_per_user_min": per_user.min(),
    "ratings_per_user_median": per_user.median(),
    "ratings_per_item_median": per_item.median(),
    "gini_item_popularity": et.gini(per_item.to_numpy()),
    "first_rating": str(pd.to_datetime(ratings.ts.min(), unit="s").date()),
    "last_rating": str(pd.to_datetime(ratings.ts.max(), unit="s").date()),
})
display(pd.Series(summary, dtype=object))

print("duplicate (user, movie) pairs:", ratings.duplicated(["user", "item"]).sum())
print("missing values in ratings    :", ratings.isna().sum().sum())
print("rating values                :", sorted(ratings.rating.unique()))
print("movies with no year in title :", items.year.isna().sum())
print("duplicate titles (diff. ids) :", items.title.duplicated().sum())
print("genres:", GENRES)
items[["item", "movieId", "title", "year"] + GENRES[:5]].head()

## 3. Ratings

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ratings.rating.value_counts().sort_index().plot.bar(ax=axes[0], color="steelblue")
axes[0].set(xlabel="rating", ylabel="number of ratings", title="Rating distribution")
stats = ratings.groupby("item").rating.agg(["size", "mean"])
corr = np.corrcoef(np.log(stats["size"]), stats["mean"])[0, 1]
axes[1].scatter(stats["size"], stats["mean"], s=5, alpha=0.3)
axes[1].set(xscale="log", xlabel="number of ratings (log)", ylabel="mean rating",
            title=f"Popularity vs mean rating (corr with log count = {corr:.2f})")
save_fig(fig, "01_ratings")
plt.show()
summary["corr_logpop_meanrating"] = corr

Things to note when you write this up: how steep the tail is, what share of ratings the top 10% of movies get, how many movies were rated only once, and whether popular movies are also rated higher (if they are, an accuracy-optimised model has a good reason to favour them, which is exactly why accuracy alone can't tell us whether popularity bias is a problem). Half-star ratings: a test movie counts as **relevant** if rated ≥ 4.0.

## 4. Genres and release years

In [ ]:
rated = ratings.merge(items[["item"] + GENRES], on="item")
genre_table = pd.DataFrame({
    "movies": items[GENRES].sum(),
    "ratings": rated[GENRES].sum(),
    "mean_rating": [rated.loc[rated[g] == 1, "rating"].mean() for g in GENRES],
}).sort_values("ratings", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
genre_table["movies"].sort_values().plot.barh(ax=axes[0], color="seagreen", title="Movies per genre")
genre_table["ratings"].sort_values().plot.barh(ax=axes[1], color="slateblue", title="Ratings per genre")
save_fig(fig, "01_genres")
plt.show()
print("genres per movie: mean", G.sum(1).mean().round(2), "| max", int(G.sum(1).max()))
genre_table.round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
items.year.dropna().astype(int).plot.hist(bins=range(1900, 2020), ax=axes[0], color="gray")
axes[0].set(xlabel="release year", title="Movies by release year")
year_of_rating = ratings.merge(items[["item", "year"]], on="item").year.dropna().astype(int)
year_of_rating.plot.hist(bins=range(1900, 2020), ax=axes[1], color="indianred")
axes[1].set(xlabel="release year", title="Ratings by movie release year")
pd.to_datetime(ratings.ts, unit="s").dt.year.value_counts().sort_index().plot.bar(ax=axes[2], color="teal")
axes[2].set(xlabel="year the rating was given", title="Ratings per year")
save_fig(fig, "01_years")
plt.show()

## 5. Feature engineering

- **Rating matrix** R (users × movies, sparse): input for Linear Regression, KNN and SVD.
- **Genre matrix** G (movies × genres, 0/1): used for genre diversity (ILD) and by the oracle in notebook 04.
- **Content features** F = genre flags + release-decade one-hot: input to PCA for the content-based model.
- **Popularity** = number of ratings per movie (computed on the training split only, inside each notebook).

In [ ]:
from sklearn.decomposition import PCA

F = et.item_features(items, GENRES)
print("content feature matrix F:", F.shape, f"({len(GENRES)} genres + decade one-hots)")
pca = PCA().fit(F)
cum = np.cumsum(pca.explained_variance_ratio_)
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(range(1, len(cum) + 1), cum, marker="o")
ax.axhline(0.9, ls="--", c="gray")
ax.set(xlabel="number of components", ylabel="cumulative explained variance",
       title="PCA on content features")
save_fig(fig, "01_pca_content")
plt.show()
for target in (0.8, 0.9):
    print(f"components for {target:.0%} variance:", int(np.searchsorted(cum, target) + 1))

## 6. Train / validation / test split

For each user, a random 20% of their ratings go to **test**. From the remaining 80% (**train**), another 20% per user is held out as **validation**. Hyperparameters are chosen on validation only; test is touched once per model at the very end. After filtering, every user still has at least about a dozen ratings, so every user appears in all three parts. Notebooks 02–04 repeat this with seeds 0, 1, 2.

In [ ]:
train, test = et.split_per_user(ratings, 0.2, seed=0)
fit, val = et.split_per_user(train, 0.2, seed=100)
for name, part in [("train", train), ("  fit", fit), ("  val", val), ("test", test)]:
    print(f"{name:6s} {len(part):6d} ratings | {part.user.nunique()} users | "
          f"min per user {part.groupby('user').size().min()}")
assert len(train) + len(test) == len(ratings)
assert set(train.user) == set(test.user) == set(range(et.N_USERS))
cold = set(test.item) - set(train.item)
print("movies that appear only in test (no model can recommend them):", len(cold))

## 7. Export for the demo site and save the summary

In [ ]:
movies = []
for _, row in items.iterrows():
    i = int(row["item"])
    movies.append({
        "id": i,
        "movieId": int(row["movieId"]),
        "title": row["title"],
        "year": None if pd.isna(row["year"]) else int(row["year"]),
        "genres": [g for g in GENRES if row[g] == 1],
        "n_ratings": int(per_item.get(i, 0)),
        "mean_rating": et.json_num(stats["mean"].get(i, np.nan), 2),
    })
et.save_json({"genres": GENRES, "movies": movies}, f"{OUT}/demo_data/movies.json")
et.save_json({k: (v if isinstance(v, str) else et.json_num(v)) for k, v in summary.items()},
             f"{OUT}/results/eda_summary.json")
print("saved", len(movies), "movies ->", f"{OUT}/demo_data/movies.json")
print(sorted(os.listdir(f"{OUT}/figures")))